# Advanced Self-RAG: Enhanced Architecture

## Improvements over Traditional Self-RAG:
1. **Uncertainty-Aware Control** — Routes questions: answer directly / retrieve / abstain (no wasted retrieval)
2. **Planner-Based Reasoning** — Decomposes complex questions into sub-steps before retrieval (replaces uncontrolled loops)
3. **Multi-Hop Targeted Retrieval** — Retrieves once per reasoning sub-step using planner output as targeted queries
4. **Bounded Refinement** — Exactly ONE refinement pass triggered by external verifier (no infinite loops)
5. **Hybrid Retrieval (NEW — Part 1)** — BM25 sparse + Dense vector retrieval fused via Reciprocal Rank Fusion (RRF)

### Architecture Flow:
```
Question
   │
   ▼
Uncertainty Controller ──► answer directly (high confidence)
   │                  ──► abstain (out of domain)
   ▼ (retrieve)
Reasoning Planner  →  [sub-query-1, sub-query-2, ...]
   │
   ▼
Hybrid Retriever (BM25 + Dense + RRF Fusion)  →  step-wise evidence per sub-query
   │
   ▼
Generator  →  draft answer
   │
   ▼
External Verifier  →  pass / refine-once
   │
   ▼
Bounded Refiner (max 1 pass)  →  final answer
```

## 1. Setup & Imports

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

import warnings
warnings.filterwarnings("ignore")

import json
import re
import math
from typing import List, Optional, Literal
from typing_extensions import TypedDict

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain import hub
from langgraph.graph import END, StateGraph, START

print("All imports successful.")

# ── PART 1: Hybrid Retrieval additional imports
from rank_bm25 import BM25Okapi                     # pip install rank-bm25
import numpy as np
import faiss                                          # pip install faiss-cpu
from langchain_community.vectorstores import FAISS as LangchainFAISS

print("All imports successful.")


## 2. Embeddings, LLM & Vector Store (Unchanged from original)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatGroq(model_name="Gemma2-9b-It")

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Vectorstore ready. {len(doc_splits)} chunks indexed.")

## 3. NEW: Uncertainty-Aware Controller

**WHY:** Traditional Self-RAG always retrieves. This wastes tokens and adds latency for questions the LLM can answer confidently from parametric knowledge, or questions completely outside the corpus domain.

**Decision:** `answer_directly` → `retrieve` → `abstain`

In [ ]:
class ControlDecision(BaseModel):
    """Routing decision for the uncertainty-aware controller."""
    decision: Literal["answer_directly", "retrieve", "abstain"] = Field(
        description="""
        'answer_directly': LLM has enough knowledge, no retrieval needed.
        'retrieve': Question requires document retrieval from the knowledge base.
        'abstain': Question is completely outside the knowledge base domain; cannot answer.
        """
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0"
    )
    reasoning: str = Field(
        description="Brief reasoning for this routing decision"
    )

uncertainty_controller_llm = llm.with_structured_output(ControlDecision)

controller_system = """
You are an intelligent routing controller for a RAG (Retrieval-Augmented Generation) system.

The knowledge base contains ONLY documents about:
- LLM-powered autonomous agents (planning, memory, tool use)
- Prompt engineering techniques
- Adversarial attacks on LLMs

Given a user question, decide:

1. 'answer_directly' — if this is general knowledge you can answer confidently without documents
   Examples: basic definitions, simple factual questions about well-known concepts

2. 'retrieve' — if the question relates to the knowledge base topics above and needs document evidence
   Examples: specific technical details about agents, prompt techniques, LLM memory types

3. 'abstain' — if the question is completely unrelated to the knowledge base AND requires specific
   factual knowledge you don't reliably have
   Examples: sports results, political figures, geography unrelated to AI

Be decisive. Provide a confidence score 0.0-1.0 and brief reasoning.
"""

controller_prompt = ChatPromptTemplate.from_messages([
    ("system", controller_system),
    ("human", "User question: {question}")
])

uncertainty_controller = controller_prompt | uncertainty_controller_llm

# Quick test
test = uncertainty_controller.invoke({"question": "what is an AI agent?"})
print(f"Decision: {test.decision} | Confidence: {test.confidence} | Reason: {test.reasoning}")

## 4. NEW: Reasoning Planner

**WHY:** Traditional Self-RAG does implicit, uncontrolled iterative reasoning. This planner makes reasoning *explicit* and *deterministic* — decomposing complex questions into ordered sub-steps BEFORE retrieval begins. This replaces the uncontrolled refinement loop with a structured plan.

**Output:** A list of sub-queries, each of which becomes a targeted retrieval query.

In [ ]:
class ReasoningPlan(BaseModel):
    """Structured reasoning plan decomposing a question into retrieval sub-steps."""
    is_simple: bool = Field(
        description="True if question is simple enough to answer in one retrieval step"
    )
    sub_queries: List[str] = Field(
        description="Ordered list of sub-queries. For simple questions, just one entry."
    )
    reasoning_strategy: str = Field(
        description="Brief description of the reasoning strategy to apply"
    )

planner_llm = llm.with_structured_output(ReasoningPlan)

planner_system = """
You are a reasoning planner for a multi-hop question answering system.

Your job: decompose a complex question into ordered sub-queries that can each be 
answered by a document retrieval step.

Rules:
- For SIMPLE questions (single fact, single concept): return is_simple=True, one sub_query.
- For COMPLEX/MULTI-HOP questions: return is_simple=False, 2-4 focused sub_queries.
- Each sub_query should be self-contained and retrieval-friendly.
- sub_queries should be ordered logically (earlier steps may inform later steps).
- Do NOT include sub_queries that are irrelevant to the knowledge base.

Example:
Question: "How does an LLM agent use memory and tools together to solve tasks?"
sub_queries:
  1. "What are the types of memory in LLM agents?"
  2. "How do LLM agents use external tools and APIs?"
  3. "How do memory and tool use work together in agent task solving?"
"""

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", planner_system),
    ("human", "Question to decompose: {question}")
])

reasoning_planner = planner_prompt | planner_llm

# Test
plan = reasoning_planner.invoke({"question": "How does an LLM agent use memory and tools together?"})
print(f"Simple: {plan.is_simple}")
print(f"Strategy: {plan.reasoning_strategy}")
print(f"Sub-queries:")
for i, sq in enumerate(plan.sub_queries, 1):
    print(f"  {i}. {sq}")

## 5. Multi-Hop Targeted Retriever

**WHY:** Traditional Self-RAG retrieves once with the original question, missing evidence needed for multi-hop reasoning. This module retrieves once *per sub-query* from the planner, producing step-indexed evidence. Each retrieval is targeted and purposeful.

In [ ]:
def multi_hop_retrieve(sub_queries: List[str], retriever, k_per_step: int = 3) -> dict:  # retriever = hybrid_retriever (BM25+Dense+RRF) — see §5b
    """
    Retrieve documents for each sub-query from the planner.
    Returns a dict mapping step index → list of documents.
    Also deduplicates across steps to avoid redundant context.
    """
    step_evidence = {}
    seen_content = set()  # deduplication by content hash
    
    for step_idx, sub_query in enumerate(sub_queries):
        raw_docs = retriever.invoke(sub_query)
        
        # Deduplicate within and across steps
        unique_docs = []
        for doc in raw_docs:
            content_hash = hash(doc.page_content[:200])  # hash first 200 chars
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                unique_docs.append(doc)
            if len(unique_docs) >= k_per_step:
                break
        
        step_evidence[step_idx] = {
            "sub_query": sub_query,
            "documents": unique_docs
        }
        print(f"  Step {step_idx+1}: '{sub_query}' → {len(unique_docs)} unique docs retrieved")
    
    return step_evidence


def flatten_evidence(step_evidence: dict) -> List:
    """Flatten step-indexed evidence into a single document list for generation."""
    all_docs = []
    for step_data in step_evidence.values():
        all_docs.extend(step_data["documents"])
    return all_docs


# Test
print("Testing multi-hop retriever:")
test_queries = ["types of memory in LLM agents", "how LLM agents use external tools"]
evidence = multi_hop_retrieve(test_queries, hybrid_retriever)  # PART 1: hybrid
print(f"Total unique docs: {sum(len(v['documents']) for v in evidence.values())}")

## 5b. PART 1 — Hybrid Retrieval: BM25 + Dense + RRF Fusion

### Why Traditional Single-Retriever Is Insufficient

| Retriever Type | Strength | Weakness |
|---|---|---|
| Dense (FAISS/Chroma) | Semantic similarity, synonym matching | Misses exact keyword matches |
| Sparse BM25 (Elasticsearch) | Exact-term, rare-word precision | Misses paraphrases / semantics |
| **Hybrid (BM25 + Dense + RRF)** | Best of both worlds | Slightly higher latency |

### Integration Strategy
```
Sub-query
    │
    ├──► BM25 sparse search  ──► ranked list A  ─┐
    │                                             ├──► RRF Fusion ──► top-k fused docs
    └──► Dense vector search ──► ranked list B  ─┘
```

### Chunking Strategy
- **Chunk size**: 100 tokens with 50-token overlap (already set in original notebook)
- **Rationale**: Smaller chunks improve BM25 term-frequency precision; overlap preserves context across boundaries
- **For BM25**: Each chunk is tokenised (whitespace split) to build the inverted index

### Top-k Selection Strategy
- BM25 fetches top-`k_sparse` candidates (default 10)
- Dense fetches top-`k_dense` candidates (default 10)
- RRF merges both lists → final top-`k_final` (default 5)
- Downstream multi-hop retriever uses `k_per_step=3` from the fused result

### Reranking Strategy
- **Reciprocal Rank Fusion (RRF)** score: `1 / (rank + k_rrf)` where `k_rrf=60` (standard constant)
- Documents appearing in BOTH lists get scores summed → naturally rewarded
- Optional cross-encoder reranking (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`) can be added as a final pass for latency-tolerant use-cases

### Elasticsearch / OpenSearch vs In-Process BM25
- **Production**: Use Elasticsearch or OpenSearch for BM25 — they scale to millions of docs and support distributed queries
- **Notebook / dev**: `rank_bm25.BM25Okapi` runs in-process, zero infra — drop-in for prototyping
- The `HybridRetriever` class below abstracts both; swap `_bm25_search` to call ES if needed

In [ ]:
# ═══════════════════════════════════════════════════════
# PART 1 — HYBRID RETRIEVAL: BM25 + Dense Vector + RRF
# ═══════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────
# 1. Build BM25 index from the same doc_splits used by Chroma
# ─────────────────────────────────────────────────────

class BM25Index:
    """
    Thin wrapper around BM25Okapi that preserves the original
    LangChain Document objects alongside the tokenised corpus.

    Production note: replace _search() body with an
    Elasticsearch / OpenSearch multi_match query for scale.
    """

    def __init__(self, documents: list):
        self.documents = documents
        # BM25 needs a tokenised corpus (list-of-lists)
        tokenised = [doc.page_content.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenised)
        print(f"  BM25Index built over {len(documents)} chunks.")

    def search(self, query: str, k: int = 10) -> list:
        """
        Return top-k (score, document) pairs sorted descending.
        Uses BM25Okapi.get_scores() — O(vocab) per query.
        """
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)           # ndarray of shape (n_docs,)
        top_indices = np.argsort(scores)[::-1][:k]      # descending order
        return [(scores[i], self.documents[i]) for i in top_indices if scores[i] > 0]


# ─────────────────────────────────────────────────────
# 2. Reciprocal Rank Fusion
# ─────────────────────────────────────────────────────

def reciprocal_rank_fusion(
    ranked_lists: list[list],   # list of ranked doc lists (each list: [doc, doc, ...])
    k_rrf: int = 60,            # standard RRF constant (60 recommended by Cormack et al.)
) -> list:
    """
    Merge N ranked lists into a single fused ranking using RRF.

    RRF score for doc d: Σ  1 / (rank(d, list_i) + k_rrf)
    Documents not appearing in a list get no contribution from that list.

    Args:
        ranked_lists: Each inner list is an ordered sequence of LangChain Documents.
        k_rrf: RRF constant. Higher → less sensitive to top ranks. 60 is canonical.

    Returns:
        List of Documents sorted by descending fused RRF score.
    """
    from collections import defaultdict

    # Use page_content[:200] as a stable document identity key
    rrf_scores: dict[str, float] = defaultdict(float)
    doc_map: dict[str, object] = {}   # key → Document object

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            key = doc.page_content[:200]
            rrf_scores[key] += 1.0 / (rank + k_rrf)
            doc_map[key] = doc   # keep the object reference

    # Sort by fused score descending
    sorted_keys = sorted(rrf_scores, key=lambda k: rrf_scores[k], reverse=True)
    return [doc_map[k] for k in sorted_keys]


# ─────────────────────────────────────────────────────
# 3. HybridRetriever — combines BM25 + Dense + RRF
# ─────────────────────────────────────────────────────

class HybridRetriever:
    """
    Drop-in replacement for LangChain's BaseRetriever that fuses
    BM25 sparse retrieval with dense vector retrieval via RRF.

    Keyword arguments
    -----------------
    bm25_index       : BM25Index — in-process BM25 (swap for ES client in prod)
    dense_retriever  : LangChain retriever backed by Chroma / FAISS / Pinecone
    k_sparse         : candidates fetched from BM25 before fusion (default 10)
    k_dense          : candidates fetched from dense store before fusion (default 10)
    k_final          : final top-k returned after RRF fusion (default 5)
    k_rrf            : RRF constant (default 60)
    """

    def __init__(
        self,
        bm25_index: BM25Index,
        dense_retriever,
        k_sparse: int = 10,
        k_dense: int = 10,
        k_final: int = 5,
        k_rrf: int = 60,
    ):
        self.bm25_index = bm25_index
        self.dense_retriever = dense_retriever
        self.k_sparse = k_sparse
        self.k_dense = k_dense
        self.k_final = k_final
        self.k_rrf = k_rrf

    # ── internal helpers ──────────────────────────────

    def _bm25_search(self, query: str) -> list:
        """
        BM25 sparse search — returns ranked list of Documents.

        Production swap:
            Replace the body below with an Elasticsearch call:
            es.search(index="my_index", query={"multi_match": {"query": query, ...}})
        """
        results = self.bm25_index.search(query, k=self.k_sparse)
        return [doc for _score, doc in results]          # strip scores; RRF uses rank only

    def _dense_search(self, query: str) -> list:
        """
        Dense semantic search via LangChain retriever (Chroma / FAISS / Pinecone).

        Production swap:
            Configure Pinecone retriever or FAISS IVFFlat for ANN at scale.
        """
        return self.dense_retriever.invoke(query)        # already returns ranked Documents

    # ── public API ────────────────────────────────────

    def invoke(self, query: str) -> list:
        """
        Run hybrid search and return top-k_final fused Documents.

        Pipeline:
          1. BM25 sparse search   → ranked_sparse  (k_sparse docs)
          2. Dense vector search  → ranked_dense   (k_dense docs)
          3. RRF fusion           → fused_ranking  (unlimited)
          4. Truncate to k_final  → final result
        """
        ranked_sparse = self._bm25_search(query)
        ranked_dense  = self._dense_search(query)

        fused = reciprocal_rank_fusion(
            [ranked_sparse, ranked_dense],
            k_rrf=self.k_rrf,
        )
        return fused[:self.k_final]


# ─────────────────────────────────────────────────────
# 4. Instantiate — Build BM25 + Hybrid retriever
# ─────────────────────────────────────────────────────
print("Building BM25 index...")
bm25_index = BM25Index(doc_splits)

print("Instantiating HybridRetriever (BM25 + Chroma dense + RRF)...")
hybrid_retriever = HybridRetriever(
    bm25_index=bm25_index,
    dense_retriever=retriever,      # the Chroma vectorstore retriever from §2
    k_sparse=10,                    # BM25 candidate pool
    k_dense=10,                     # Dense candidate pool
    k_final=5,                      # Final fused top-k per query
    k_rrf=60,                       # RRF constant
)

# ── Quick smoke-test
print("\nSmoke-test — query: 'types of memory in LLM agents'")
test_docs = hybrid_retriever.invoke("types of memory in LLM agents")
print(f"  → {len(test_docs)} fused documents returned")
for i, d in enumerate(test_docs, 1):
    print(f"  [{i}] {d.page_content[:80].strip()}...")

print("\nHybrid retriever ready.")


## 6. Generator (Kept from original, now uses aggregated multi-hop evidence)

In [ ]:
rag_prompt = hub.pull("rlm/rag-prompt")
rag_chain = rag_prompt | llm

print("Generator ready.")

## 7. NEW: External Verifier

**WHY:** Traditional Self-RAG uses self-verification — the same model that generated the answer also verifies it, creating self-bias and systematic blind spots. The external verifier acts as an *independent* judge, checking:
1. Factual grounding in retrieved evidence
2. Whether the answer actually addresses the question
3. What specific evidence gaps exist (to guide bounded refinement)

In [ ]:
class VerificationResult(BaseModel):
    """External verifier output with detailed feedback."""
    factual_grounding: Literal["supported", "partial", "unsupported"] = Field(
        description="Is the answer grounded in the retrieved evidence?"
    )
    answers_question: bool = Field(
        description="Does the answer actually address the user's question?"
    )
    verdict: Literal["pass", "refine"] = Field(
        description="'pass' = acceptable answer, 'refine' = needs one refinement pass"
    )
    evidence_gaps: List[str] = Field(
        description="List of specific missing pieces of evidence (empty if verdict=pass)"
    )
    confidence: float = Field(
        description="Verifier confidence in this assessment, 0.0-1.0"
    )

verifier_llm = llm.with_structured_output(VerificationResult)

verifier_system = """
You are an independent factual verifier for a RAG question-answering system.
You are NOT the model that generated the answer — you assess it objectively.

Your job:
1. Check if the generated answer is GROUNDED in the provided evidence documents.
   - 'supported': all key claims have evidence backing
   - 'partial': some claims supported, some unsupported or hallucinated
   - 'unsupported': answer makes claims not found in evidence

2. Check if the answer actually ADDRESSES the user's question.

3. Decide:
   - 'pass': answer is acceptable (grounded + addresses question)
   - 'refine': needs exactly ONE more retrieval-refinement pass

4. If verdict='refine', list specific evidence_gaps (what's missing).
   These gaps will be used as retrieval queries in the refinement pass.

Be strict about hallucinations. If the answer introduces facts not in evidence, mark 'refine'.
"""

verifier_prompt = ChatPromptTemplate.from_messages([
    ("system", verifier_system),
    ("human", """
User Question: {question}

Retrieved Evidence:
{evidence}

Generated Answer:
{answer}

Verify this answer.
""")
])

external_verifier = verifier_prompt | verifier_llm

print("External verifier ready.")

## 8. NEW: Bounded Refiner (Max 1 Pass)

**WHY:** Traditional Self-RAG loops indefinitely until satisfied, causing latency explosion and potential infinite loops (as seen in your original code hitting Groq's rate limit). This bounded refiner does exactly ONE targeted refinement pass, retrieving only for the specific evidence gaps identified by the verifier.

In [ ]:
refinement_system = """
You are a refinement specialist for a question-answering system.
You are given:
- The original question
- A draft answer that needs improvement  
- The original evidence used
- NEW additional evidence retrieved specifically for the identified gaps
- The specific gaps that need to be addressed

Your task: produce an IMPROVED final answer that:
1. Fixes the identified gaps using the new evidence
2. Keeps correct parts of the draft answer
3. Stays strictly grounded in the combined evidence
4. Is concise (3-5 sentences max)

Do NOT add information not found in any of the evidence.
"""

refinement_prompt = ChatPromptTemplate.from_messages([
    ("system", refinement_system),
    ("human", """
Original Question: {question}

Draft Answer: {draft_answer}

Original Evidence:
{original_evidence}

Additional Evidence (for gaps):
{additional_evidence}

Evidence Gaps to Address: {gaps}

Write the improved final answer:
""")
])

refiner_chain = refinement_prompt | llm | StrOutputParser()

print("Bounded refiner ready.")

## 9. AgentState — Extended for Advanced Architecture

**Changes from original:** Added `control_decision`, `reasoning_plan`, `step_evidence`, `verification_result`, `refinement_done` fields. The `refinement_done` flag is the key guard preventing infinite loops.

In [ ]:
class AdvancedAgentState(TypedDict):
    # Core fields (from original)
    question: str
    documents: List          # All retrieved documents (flattened)
    generation: str          # Final answer string
    
    # NEW: Uncertainty controller output
    control_decision: str    # 'answer_directly' | 'retrieve' | 'abstain'
    control_confidence: float
    
    # NEW: Planner output
    sub_queries: List[str]   # Ordered list of retrieval sub-queries
    is_simple_question: bool
    
    # NEW: Step-indexed evidence from multi-hop retrieval
    step_evidence: dict      # {step_idx: {sub_query, documents}}
    
    # NEW: Verifier output
    verification_verdict: str    # 'pass' | 'refine'
    evidence_gaps: List[str]     # Gaps to address in refinement
    
    # NEW: Bounded refinement guard
    refinement_done: bool    # True after one refinement pass — prevents loops

print("AdvancedAgentState defined.")

## 10. Node Functions — Advanced Pipeline

In [ ]:
# ─────────────────────────────────────────────
# NODE 1: Uncertainty Controller
# NEW: Routes question before any retrieval happens
# ─────────────────────────────────────────────
def run_uncertainty_controller(state: AdvancedAgentState):
    print("\n──── [1] UNCERTAINTY CONTROLLER ────")
    question = state["question"]
    
    result = uncertainty_controller.invoke({"question": question})
    
    print(f"  Decision   : {result.decision}")
    print(f"  Confidence : {result.confidence:.2f}")
    print(f"  Reasoning  : {result.reasoning}")
    
    return {
        "control_decision": result.decision,
        "control_confidence": result.confidence,
        "question": question,
        "refinement_done": False,  # initialize
    }


# ─────────────────────────────────────────────
# NODE 2: Reasoning Planner
# NEW: Decomposes question into sub-queries before retrieval
# ─────────────────────────────────────────────
def run_reasoning_planner(state: AdvancedAgentState):
    print("\n──── [2] REASONING PLANNER ────")
    question = state["question"]
    
    plan = reasoning_planner.invoke({"question": question})
    
    print(f"  Simple question : {plan.is_simple}")
    print(f"  Strategy        : {plan.reasoning_strategy}")
    print(f"  Sub-queries ({len(plan.sub_queries)}):")
    for i, sq in enumerate(plan.sub_queries, 1):
        print(f"    {i}. {sq}")
    
    return {
        "sub_queries": plan.sub_queries,
        "is_simple_question": plan.is_simple,
    }


# ─────────────────────────────────────────────
# NODE 3: Multi-Hop Targeted Retrieval
# NEW: Retrieves once per sub-query, deduplicates
# ─────────────────────────────────────────────
def run_multi_hop_retrieval(state: AdvancedAgentState):
    print("\n──── [3] MULTI-HOP RETRIEVAL ────")
    sub_queries = state["sub_queries"]
    
    step_evidence = multi_hop_retrieve(sub_queries, hybrid_retriever, k_per_step=3)  # PART 1: using hybrid BM25+Dense+RRF retriever
    all_docs = flatten_evidence(step_evidence)
    
    print(f"  Total unique docs after dedup: {len(all_docs)}")
    
    return {
        "step_evidence": step_evidence,
        "documents": all_docs,
    }


# ─────────────────────────────────────────────
# NODE 4: Generator
# Generates answer from aggregated multi-hop evidence
# ─────────────────────────────────────────────
def run_generator(state: AdvancedAgentState):
    print("\n──── [4] GENERATOR ────")
    question = state["question"]
    documents = state["documents"]
    
    generation = rag_chain.invoke({"context": documents, "question": question})
    generation_text = generation.content if hasattr(generation, "content") else str(generation)
    
    print(f"  Draft answer: {generation_text[:150]}...")
    
    return {"generation": generation_text}


# ─────────────────────────────────────────────
# NODE 5: External Verifier
# NEW: Independent verification — no self-bias
# ─────────────────────────────────────────────
def run_external_verifier(state: AdvancedAgentState):
    print("\n──── [5] EXTERNAL VERIFIER ────")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    
    # Format evidence for verifier
    evidence_text = "\n---\n".join(
        [f"[Doc {i+1}]: {doc.page_content}" for i, doc in enumerate(documents[:6])]
    )
    
    result = external_verifier.invoke({
        "question": question,
        "evidence": evidence_text,
        "answer": generation,
    })
    
    print(f"  Factual grounding : {result.factual_grounding}")
    print(f"  Answers question  : {result.answers_question}")
    print(f"  Verdict           : {result.verdict}")
    print(f"  Confidence        : {result.confidence:.2f}")
    if result.evidence_gaps:
        print(f"  Evidence gaps:")
        for gap in result.evidence_gaps:
            print(f"    - {gap}")
    
    return {
        "verification_verdict": result.verdict,
        "evidence_gaps": result.evidence_gaps,
    }


# ─────────────────────────────────────────────
# NODE 6: Bounded Refiner
# NEW: ONE targeted refinement pass, then STOP
# The refinement_done flag prevents any second pass
# ─────────────────────────────────────────────
def run_bounded_refiner(state: AdvancedAgentState):
    print("\n──── [6] BOUNDED REFINER (1 pass only) ────")
    question = state["question"]
    generation = state["generation"]
    documents = state["documents"]
    evidence_gaps = state["evidence_gaps"]
    
    # Retrieve specifically for the evidence gaps
    additional_docs = []
    print(f"  Retrieving for {len(evidence_gaps)} evidence gap(s):")
    for gap in evidence_gaps:
        print(f"    Gap query: '{gap}'")
        gap_docs = hybrid_retriever.invoke(gap)  # PART 1: hybrid retrieval for gap filling
        additional_docs.extend(gap_docs[:2])  # max 2 docs per gap
    
    # Deduplicate additional docs against existing
    existing_hashes = {hash(d.page_content[:200]) for d in documents}
    new_docs = [d for d in additional_docs 
                if hash(d.page_content[:200]) not in existing_hashes]
    
    print(f"  New unique docs for refinement: {len(new_docs)}")
    
    # Format evidence
    original_evidence_text = "\n---\n".join(
        [doc.page_content for doc in documents[:4]]
    )
    additional_evidence_text = "\n---\n".join(
        [doc.page_content for doc in new_docs]
    ) if new_docs else "No additional evidence found."
    
    refined = refiner_chain.invoke({
        "question": question,
        "draft_answer": generation,
        "original_evidence": original_evidence_text,
        "additional_evidence": additional_evidence_text,
        "gaps": ", ".join(evidence_gaps) if evidence_gaps else "general improvement",
    })
    
    print(f"  Refined answer: {refined[:150]}...")
    
    # Update documents with new evidence
    combined_docs = documents + new_docs
    
    return {
        "generation": refined,
        "documents": combined_docs,
        "refinement_done": True,  # CRITICAL: prevents any further refinement
    }


# ─────────────────────────────────────────────
# NODE 7: Direct Answer (no retrieval needed)
# ─────────────────────────────────────────────
def run_direct_answer(state: AdvancedAgentState):
    print("\n──── [DIRECT ANSWER — no retrieval needed] ────")
    question = state["question"]
    
    direct_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Answer the question concisely from your knowledge."),
        ("human", "{question}")
    ])
    chain = direct_prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question})
    
    print(f"  Direct answer: {answer[:150]}")
    return {"generation": answer, "documents": [], "refinement_done": True}


# ─────────────────────────────────────────────
# NODE 8: Abstain
# ─────────────────────────────────────────────
def run_abstain(state: AdvancedAgentState):
    print("\n──── [ABSTAIN — out of domain] ────")
    return {
        "generation": "I cannot answer this question as it falls outside the scope of my knowledge base, which covers LLM agents and prompt engineering.",
        "documents": [],
        "refinement_done": True,
    }


print("All node functions defined.")

## 11. Conditional Edge Functions

In [ ]:
def route_by_controller(state: AdvancedAgentState) -> str:
    """Routes after uncertainty controller."""
    decision = state["control_decision"]
    print(f"  → Routing: {decision}")
    
    if decision == "answer_directly":
        return "direct_answer"
    elif decision == "abstain":
        return "abstain"
    else:  # retrieve
        return "planner"


def route_after_verification(state: AdvancedAgentState) -> str:
    """
    Routes after external verifier.
    KEY: if refinement already done once, ALWAYS go to END.
    This is the bounded refinement guard.
    """
    verdict = state["verification_verdict"]
    refinement_done = state.get("refinement_done", False)
    
    if refinement_done:
        print("  → Refinement already done. Going to END (bounded).")
        return "end"
    
    if verdict == "pass":
        print("  → Verification PASSED. Going to END.")
        return "end"
    else:  # refine
        print("  → Verification: REFINE (one bounded pass).")
        return "bounded_refiner"


print("Edge functions defined.")

## 12. Build the Advanced LangGraph Workflow

In [ ]:
workflow = StateGraph(AdvancedAgentState)

# ── Add all nodes
workflow.add_node("uncertainty_controller", run_uncertainty_controller)
workflow.add_node("planner",               run_reasoning_planner)
workflow.add_node("multi_hop_retriever",   run_multi_hop_retrieval)
workflow.add_node("generator",             run_generator)
workflow.add_node("external_verifier",     run_external_verifier)
workflow.add_node("bounded_refiner",       run_bounded_refiner)
workflow.add_node("direct_answer",         run_direct_answer)
workflow.add_node("abstain",               run_abstain)

# ── Entry point
workflow.add_edge(START, "uncertainty_controller")

# ── Controller routes to: direct_answer / abstain / planner
workflow.add_conditional_edges(
    "uncertainty_controller",
    route_by_controller,
    {
        "direct_answer": "direct_answer",
        "abstain":       "abstain",
        "planner":       "planner",
    }
)

# ── Planner → Multi-hop retriever → Generator → Verifier
workflow.add_edge("planner",             "multi_hop_retriever")
workflow.add_edge("multi_hop_retriever", "generator")
workflow.add_edge("generator",           "external_verifier")

# ── Verifier routes to: END (pass) / bounded_refiner (refine once)
workflow.add_conditional_edges(
    "external_verifier",
    route_after_verification,
    {
        "end":             END,
        "bounded_refiner": "bounded_refiner",
    }
)

# ── Bounded refiner → Verifier again (but refinement_done=True so it goes to END)
workflow.add_edge("bounded_refiner", "external_verifier")

# ── Terminal nodes
workflow.add_edge("direct_answer", END)
workflow.add_edge("abstain",       END)

# ── Compile
app = workflow.compile()

print("Advanced Self-RAG workflow compiled successfully.")

## 13. Visualize the Graph

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph(xray=True).draw_mermaid_png()))

## 14. Test Cases

### Test 1: Complex multi-hop question (full pipeline)

In [ ]:
print("=" * 60)
print("TEST 1: Complex multi-hop question")
print("=" * 60)

result = app.invoke({"question": "How do LLM agents use memory and tools together to solve complex tasks?"})

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result["generation"])

### Test 2: Simple question (direct answer, no retrieval)

In [ ]:
print("=" * 60)
print("TEST 2: Simple question (should answer directly)")
print("=" * 60)

result2 = app.invoke({"question": "What does RAG stand for?"})

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result2["generation"])
print(f"\n(Control decision: {result2.get('control_decision')} — retrieval skipped: {len(result2.get('documents', [])) == 0})")

### Test 3: Out-of-domain question (should abstain)

In [ ]:
print("=" * 60)
print("TEST 3: Out-of-domain question (should abstain)")
print("=" * 60)

result3 = app.invoke({"question": "Who is the first president of Sri Lanka?"})

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result3["generation"])
print(f"\n(Control decision: {result3.get('control_decision')})")

### Test 4: Retrieval question (should retrieve, verify, and possibly refine once)

In [ ]:
print("=" * 60)
print("TEST 4: Technical retrieval question")
print("=" * 60)

result4 = app.invoke({"question": "What are the different types of agent memory and how does long-term memory work?"})

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result4["generation"])
print(f"\nVerification verdict : {result4.get('verification_verdict')}")
print(f"Refinement done      : {result4.get('refinement_done')}")
print(f"Total docs used      : {len(result4.get('documents', []))}")

## 15. Architecture Comparison Summary

| Feature | Traditional Self-RAG | Advanced Self-RAG |
|---|---|---|
| Retrieval trigger | Always retrieves | Uncertainty controller gates retrieval |
| Reasoning | Implicit, iterative loops | Explicit planner → structured sub-queries |
| Retrieval strategy | Single query, original question | Multi-hop: one retrieval per sub-query |
| Verification | Self-verification (same model) | Independent external verifier |
| Refinement | Unbounded loop (can hit rate limits) | Exactly 1 pass via `refinement_done` flag |
| Out-of-domain | Gets confused, re-queries | Abstains cleanly |
| Simple questions | Still retrieves | Answers directly (lower latency + cost) |
| **Retrieval method** | Dense-only (Chroma/FAISS) | **Hybrid: BM25 sparse + Dense + RRF fusion** |